# Final Project, Part 1: Solar Flare Visualization Observatory

## Dataset Information

- **Dataset:** GOES X-ray Flux from NOAA SWPC
- **Source:** NOAA Space Weather Prediction Center
- **URL:** https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json
- **Format:** JSON
- **Why:** Real-time data to explore solar flare activity


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests
import numpy as np

# Load data
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
response = requests.get(url)
data = pd.DataFrame(response.json())

# Preprocess
data['time_tag'] = pd.to_datetime(data['time_tag'])
long_flux = data[data['energy'] == '0.1-0.8nm'].copy()
long_flux_sorted = long_flux.sort_values(by='flux', ascending=False)

# Color by flare class
def flare_color(flux):
    if flux < 1e-6: return 'yellow'
    elif flux < 1e-5: return 'orange'
    elif flux < 1e-4: return 'red'
    else: return 'purple'

# Plot function
def plot_interactive_flare_map(top_n=5, show_aurora=True):
    top_flares = long_flux_sorted.head(top_n)

    fig = plt.figure(figsize=(16, 7))
    gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1])

    ax_map = plt.subplot(gs[0], projection=ccrs.PlateCarree())
    ax_map.set_global()
    ax_map.coastlines()
    ax_map.gridlines(draw_labels=True)
    ax_map.add_feature(cfeature.LAND, facecolor='lightgray')
    ax_map.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax_map.set_title(f'Top {top_n} Flare Impact Map', fontsize=14)

    for _, row in top_flares.iterrows():
        flare_time = row['time_tag']
        flux = row['flux']
        lon = -15 * (flare_time.hour + flare_time.minute / 60)
        color = flare_color(flux)
        size = np.clip(flux * 1e8, 10, 40)
        ax_map.plot(lon, 0, marker='o', color=color, markersize=size,
                    transform=ccrs.PlateCarree(), alpha=0.6)

    strongest = top_flares.iloc[0]
    subsolar_lon = -15 * (strongest['time_tag'].hour + strongest['time_tag'].minute / 60)
    ax_map.add_patch(plt.Rectangle((subsolar_lon - 60, -60), 120, 120,
                                   transform=ccrs.PlateCarree(), color='gold', alpha=0.2,
                                   label='Daylit Region'))

    if show_aurora:
        for lat in [65, -65]:
            ax_map.plot(np.linspace(-180, 180, 500), [lat]*500,
                        transform=ccrs.Geodetic(), linestyle='--',
                        color='blue', linewidth=1.2,
                        label='Aurora Oval' if lat == 65 else None)

    ax_map.legend(loc='lower left')

    ax_time = plt.subplot(gs[1])
    ax_time.plot(long_flux['time_tag'], long_flux['flux'], color='black',
                 label='X-ray Flux (0.1–0.8nm)', linewidth=1)
    ax_time.set_yscale('log')
    ax_time.set_title('X-ray Flux Time Series (log scale)', fontsize=14)
    ax_time.set_xlabel('UTC Time')
    ax_time.set_ylabel('Flux (W/m²)')
    ax_time.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    ax_time.axhline(1e-6, color='orange', linestyle='--', linewidth=1, label='C-class')
    ax_time.axhline(1e-5, color='red', linestyle='--', linewidth=1, label='M-class')
    ax_time.axhline(1e-4, color='purple', linestyle='--', linewidth=1, label='X-class')
    ax_time.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

# Call function
plot_interactive_flare_map(top_n=10, show_aurora=True);


## What Worked
- Successfully mapped flare intensity geographically
- Time series revealed flare timing clearly
- Aurora toggle was effective for user control

## What Didn't Work
- Heatmaps were unclear
- Emoji titles caused font warnings
- Some environments lacked ipywidgets support